In [ ]:
# CNN1
import tensorflow as tf
from keras import layers, models
import keras_tuner as kt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr
from keras.callbacks import EarlyStopping

# Custom callback to save overall best model across all trials
class OverallBestModelCheckpoint(tf.keras.callbacks.Callback):
    overall_best_val_loss = np.Inf  # Class-level variable to track global best

    def __init__(self, filepath, monitor='val_loss', mode='min', verbose=0):
        super(OverallBestModelCheckpoint, self).__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.verbose = verbose
        self.best_val_loss_in_trial = np.Inf  # Best in current trial

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_val_loss = logs.get(self.monitor)
        if current_val_loss is None:
            return
        
        # Update the best value for the current trial if improved
        if current_val_loss < self.best_val_loss_in_trial:
            self.best_val_loss_in_trial = current_val_loss
        
        # Compare with overall best across trials
        if self.best_val_loss_in_trial < OverallBestModelCheckpoint.overall_best_val_loss:
            OverallBestModelCheckpoint.overall_best_val_loss = self.best_val_loss_in_trial
            self.model.save(self.filepath)
            if self.verbose > 0:
                print(f'\nEpoch {epoch+1}: {self.monitor} improved to {current_val_loss:.5f} (overall), model saved.')

def build_cnn_model(hp):
    # Define two inputs: one for the CNN branch and one for the MLP branch
    input_cnn = tf.keras.layers.Input(shape=(112, 4), name='cnn_input')
    input_mlp = tf.keras.layers.Input(shape=(1,), name='mlp_input')

    # CNN branch (as in your original code)
    x = tf.keras.layers.Conv1D(
        filters=hp.Int('filters', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size', min_value=2, max_value=7, step=1),  # Kernel size smaller than or equal to input length
        activation='relu',
        padding='same'
    )(input_cnn)
    x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)  # Pooling layer (can reduce sequence length)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dropout(rate=hp.Float('dropout_rate', min_value=0.0, max_value=0.5))(x)
    x = tf.keras.layers.Dense(
        hp.Int('dense_units', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x)
    
    # MLP branch for the additional input from merged_time.txt
    y = tf.keras.layers.Dense(
        hp.Int('mlp_units', min_value=8, max_value=64, step=8),
        activation='relu'
    )(input_mlp)
    
    # Concatenate the outputs of the CNN and MLP branches
    concatenated = tf.keras.layers.Concatenate()([x, y])
    
    # Output layer for regression task (predicting a single value)
    output = tf.keras.layers.Dense(1)(concatenated)
    
    # Create and compile the model using the functional API
    model = tf.keras.models.Model(inputs=[input_cnn, input_mlp], outputs=output)
    
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG'),
        weight_decay=1e-4
    )
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=['mae']
    )
    return model

# Create a tuner for the CNN model
tuner_cnn = kt.Hyperband(
    build_cnn_model,
    objective='val_loss',  # Objective is to minimize the validation loss
    max_epochs=100,
    factor=3,
    directory='cnn1_tuning',
    project_name='cnn1_hyperparameter_search_regression'
)

# Define EarlyStopping to keep the patience mechanism (patience=10)
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor the validation loss
    patience=10,         # Stop after 10 epochs with no improvement
    restore_best_weights=True  # Restore the best model weights
)

# Instantiate the custom overall best model checkpoint callback
overall_best_checkpoint = OverallBestModelCheckpoint(
    filepath='best_CNN1.h5',
    monitor='val_loss',
    mode='min',
    verbose=1
)

log_dir = "logs_best_CNN1"
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# Open the file and read its contents for the CNN branch input
with open("Feature_CNN1.txt", 'r') as file:
    data = []  # List to hold the 2D arrays
    current_array = []  # Temporary list for the current array

    for line in file:
        line = line.strip()  

        # Check for a blank line
        if line == "":
            if current_array:  # If current_array has data, save it to data
                data.append(current_array)
                current_array = []  # Reset for the next block
        else:
            # Remove the brackets and split the line into elements
            line = line.replace('[', '').replace(']', '')  # Remove square brackets
            current_array.append([float(x.strip().replace(',', '')) for x in line.split()])

    # Append the last array if it's not empty
    if current_array:
        data.append(current_array)
        
# Convert to numpy array and reshape for CNN input
sample_data = np.array(data)
print(len(sample_data))
X = sample_data.reshape(len(sample_data), 112, 4)  

# Read MLP input from merged_time.txt (each sample has one value)
exp_cond_data = []
with open('merged_Cas13a_1e9_time_before_19min.txt', 'r') as file:
    exp_cond_data = [float(line.strip()) for line in file.readlines()]
exp_cond_data = np.array(exp_cond_data)
X_exp_cond_data = exp_cond_data.reshape(len(exp_cond_data), 1)  

# Read target values
rate = []
with open('merged_Cas13a_1e9_values_unique_before_19min.txt', 'r') as file:
    rate = [float(line.strip()) for line in file.readlines()]
y = np.array(rate)

# Define random seed to ensure input data consistency
np.random.seed(42)
# Generate 800 random indices from the range [0, len(X))
indices_to_select = np.random.choice(len(X), size=800, replace=False)
# Get all indices
all_indices = np.arange(len(X))
# Get the indices of the unselected data
indices_unselected = np.setdiff1d(all_indices, indices_to_select)

# Select elements from X, y, and mlp_data based on the random indices
X_selected = X[indices_to_select]
y_selected = y[indices_to_select]
X_exp_cond_data_selected = X_exp_cond_data[indices_to_select]

# Get unselected elements as new data to test the model
X_unselected = X[indices_unselected]
y_unselected = y[indices_unselected]
X_exp_cond_data_unselected = X_exp_cond_data[indices_unselected]

# Split the data into training and testing sets (ensuring both inputs are split)
X_train, X_test, X_exp_cond_data_train, X_exp_cond_data_test, y_train, y_test = train_test_split(
    X_selected, X_exp_cond_data_selected, y_selected, test_size=0.2, random_state=42)

# Start the hyperparameter search including EarlyStopping, TensorBoard, and our custom OverallBestModelCheckpoint callback.
tuner_cnn.search(
    [X_train, X_exp_cond_data_train], y_train, 
    validation_data=([X_test, X_exp_cond_data_test], y_test), 
    callbacks=[early_stopping, tensorboard_callback, overall_best_checkpoint]
)

# Load the saved overall best model for final evaluation.
final_model = tf.keras.models.load_model('best_CNN1.h5')

# Evaluate the final model on the test set
val_loss, val_mae = final_model.evaluate([X_test, X_exp_cond_data_test], y_test)
print(f"Final Best Model - Test Loss: {val_loss}, Test MAE: {val_mae}")

# Predict the values on the test set
y_pred = final_model.predict([X_test, X_exp_cond_data_test])
# Compute Spearman correlation between true values and predicted values
spearman_corr_test, _ = spearmanr(y_test, y_pred)
print(f"Final Best Model - Spearman Correlation for the test set: {spearman_corr_test}")



# Predict the values on the new data set, 100 independent trials using the final best model
i = 0
spearman_array = []

while i < 100:
    # Generate 100 random indices from the unselected data
    indices_to_select_new = np.random.choice(len(X_unselected), size=100, replace=False)
    # Select elements from X, y, and mlp_data based on the random indices
    X_unselected_selected = X_unselected[indices_to_select_new]
    y_unselected_selected = y_unselected[indices_to_select_new]
    X_exp_cond_data_unselected_selected = X_exp_cond_data_unselected[indices_to_select_new]

    # Use the final saved model for prediction
    y_pred = final_model.predict([X_unselected_selected, X_exp_cond_data_unselected_selected])
    # Compute Spearman correlation between true values and predicted values
    spearman_corr_new, _ = spearmanr(y_unselected_selected, y_pred)
    spearman_array.append(spearman_corr_new)
    print(f"Spearman Correlation for new data: {spearman_corr_new}")
    i += 1

for spearman_corr in spearman_array:
    print(spearman_corr)

In [ ]:
# CNN1+CNN2
from tensorflow import keras

import tensorflow as tf
from keras import layers, models
import keras_tuner as kt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr
from keras.callbacks import EarlyStopping

# Custom callback to save overall best model across all trials
class OverallBestModelCheckpoint(tf.keras.callbacks.Callback):
    overall_best_val_loss = np.Inf  # Class-level variable to track global best

    def __init__(self, filepath, monitor='val_loss', mode='min', verbose=0):
        super(OverallBestModelCheckpoint, self).__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.verbose = verbose
        self.best_val_loss_in_trial = np.Inf  # Best in current trial

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_val_loss = logs.get(self.monitor)
        if current_val_loss is None:
            return
        
        # Update the best value for the current trial if improved
        if current_val_loss < self.best_val_loss_in_trial:
            self.best_val_loss_in_trial = current_val_loss
        
        # Compare with overall best across trials
        if self.best_val_loss_in_trial < OverallBestModelCheckpoint.overall_best_val_loss:
            OverallBestModelCheckpoint.overall_best_val_loss = self.best_val_loss_in_trial
            self.model.save(self.filepath)
            if self.verbose > 0:
                print(f'\nEpoch {epoch+1}: {self.monitor} improved to {current_val_loss:.5f} (overall), model saved.')

def build_fusion_model(hp):
    """
    Builds a two-branch CNN model using the Keras Functional API.
    Each branch processes one input, and then the outputs are concatenated
    for a final prediction (regression).
    """

    # Branch 1 for input of shape (112, 4)
    input1 = keras.Input(shape=(112, 4), name='branch1_input')

    # Conv1D layer for branch 1
    x1 = layers.Conv1D(
        filters=hp.Int('filters1', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size1', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input1)
    x1 = layers.MaxPooling1D(pool_size=2)(x1)
    x1 = layers.Flatten()(x1)
    x1 = layers.Dense(
        hp.Int('dense_units1', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x1)

    # Branch 2 for input of shape (184, 3)
    input2 = keras.Input(shape=(184, 3), name='branch2_input')

    x2 = layers.Conv1D(
        filters=hp.Int('filters2', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size2', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input2)
    x2 = layers.MaxPooling1D(pool_size=2)(x2)
    x2 = layers.Flatten()(x2)
    x2 = layers.Dense(
        hp.Int('dense_units2', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x2)

    # Branch 3 for input of shape (1,)
    input3 = keras.Input(shape=(1,), name='branch3_input')

    x3 = layers.Dense(
        hp.Int('mlp_units', min_value=8, max_value=64, step=8),
        activation='relu'
    )(input3)

    # Fuse (concatenate) the outputs of both branches
    merged = layers.Concatenate()([x1, x2, x3])

    # Add dropout layer with hyperparameter dropout_rate ranging from 0 to 0.5
    dropout_rate = hp.Float('dropout_rate', min_value=0.0, max_value=0.5)
    x = layers.Dropout(rate=dropout_rate)(merged)

    # Add a dense layer after concatenation
    x = layers.Dense(
        hp.Int('dense_units_merged', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x)

    # Final output layer for regression (1 unit)
    output = layers.Dense(1, name='output')(x)

    # Build and compile the final model
    model = keras.Model(inputs=[input1, input2, input3], outputs=output)

    # Compile the model with AdamW optimizer (L2 regularization via weight decay = 1e-4) and mean squared error loss
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG'),
        weight_decay=1e-4
    )
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=['mae']
    )

    return model

# Hyperparameter Tuner Setup
tuner_fusion = kt.Hyperband(
    build_fusion_model,
    objective='val_loss',  # Minimize validation loss for regression
    max_epochs=100,
    factor=3,  # Hyperband factor
    directory='cnn1_cnn2_tuning',
    project_name='cnn1_cnn2_hyperparameter_search_regression'
)

# Data Loading and Preprocessing
# 1) Load first input data (shape = (n_samples, 112, 4))
with open("Feature_CNN1.txt", 'r') as file:
    data_branch1 = []
    current_array = []
    for line in file:
        line = line.strip()
        if line == "":
            if current_array:
                data_branch1.append(current_array)
                current_array = []
        else:
            line = line.replace('[', '').replace(']', '')
            current_array.append([float(x.strip().replace(',', '')) for x in line.split()])
    if current_array:
        data_branch1.append(current_array)

X_branch1 = np.array(data_branch1)  # shape: (n_samples, 112, 4)
X_branch1 = X_branch1.reshape(len(X_branch1), 112, 4)  

# 2) Load second input data (shape = (n_samples, 184, 3))
with open("Feature_CNN2.txt", 'r') as file:
    data_branch2 = []
    current_array2 = []
    for line in file:
        line = line.strip()
        if line == "":
            if current_array2:
                data_branch2.append(current_array2)
                current_array2 = []

        else:
            line = line.replace('[', '').replace(']', '')
            current_array2.append([float(x.strip().replace(',', '')) for x in line.split()])
    if current_array2:
        data_branch2.append(current_array2)

X_branch2 = np.array(data_branch2)  # shape: (n_samples, 184, 3)
X_branch2 = X_branch2.reshape(len(X_branch2), 184, 3)  

# Read MLP input from merged_time.txt (each sample has one value)
data_branch3 = []
with open('merged_Cas13a_1e9_time_before_19min.txt', 'r') as file:
    data_branch3 = [float(line.strip()) for line in file.readlines()]
X_branch3 = np.array(data_branch3)
X_branch3 = X_branch3.reshape(len(X_branch3), 1)  

# 3) Load regression targets (y)
rate = []
with open('merged_Cas13a_1e9_values_unique_before_19min.txt', 'r') as file:
    rate = [float(line.strip()) for line in file.readlines()]
y = np.array(rate)

# Ensure both X_branch1 and X_branch2 have the same first dimension as y
assert len(X_branch1) == len(X_branch2) == len(X_branch3) == len(y), "All inputs must have the same number of samples."

np.random.seed(42)
indices_to_select = np.random.choice(len(X_branch1), size=800, replace=False)
all_indices = np.arange(len(X_branch1))
indices_unselected = np.setdiff1d(all_indices, indices_to_select)

X1_selected = X_branch1[indices_to_select]
X2_selected = X_branch2[indices_to_select]
X3_selected = X_branch3[indices_to_select]
y_selected = y[indices_to_select]

X1_unselected = X_branch1[indices_unselected]
X2_unselected = X_branch2[indices_unselected]
X3_unselected = X_branch3[indices_unselected]
y_unselected = y[indices_unselected]

# Split the selected data into training and testing sets
X1_train, X1_test, X2_train, X2_test, X3_train, X3_test, y_train, y_test = train_test_split(
    X1_selected, 
    X2_selected, 
    X3_selected,
    y_selected,
    test_size=0.2,
    random_state=42
)

# Define callbacks: EarlyStopping, TensorBoard, etc.
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Instantiate the custom overall best model checkpoint callback
overall_best_checkpoint = OverallBestModelCheckpoint(
    filepath='best_CNN1_CNN2.h5',
    monitor='val_loss',
    mode='min',
    verbose=1
)

log_dir = "logs_best_CNN1_CNN2"
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# Run Hyperparameter Search
tuner_fusion.search(
    [X1_train, X2_train, X3_train],  
    y_train,
    validation_data=([X1_test, X2_test, X3_test], y_test),
    callbacks=[early_stopping, tensorboard_callback, overall_best_checkpoint]
)

# Load the saved overall best model for final evaluation.
final_model = tf.keras.models.load_model('best_CNN1_CNN2.h5')

# Retrieve the Best Model and Evaluate
val_loss, val_mae = final_model.evaluate([X1_test, X2_test, X3_test], y_test)
print(f"Best Fusion CNN model - Test Loss: {val_loss}, Test MAE: {val_mae}")

# Predict and compute Spearman correlation on the test set
y_pred_test = final_model.predict([X1_test, X2_test, X3_test])
spearman_corr_test, _ = spearmanr(y_test, y_pred_test)
print(f"Spearman Correlation (test set): {spearman_corr_test}")



# Evaluate on new (unselected) data multiple times 
spearman_array = []
for i in range(100):
    # Randomly select 100 samples from the unselected portion
    indices_to_select_new = np.random.choice(len(X1_unselected), size=100, replace=False)

    X1_unselected_selected = X1_unselected[indices_to_select_new]
    X2_unselected_selected = X2_unselected[indices_to_select_new]
    X3_unselected_selected = X3_unselected[indices_to_select_new]
    y_unselected_selected = y_unselected[indices_to_select_new]

    # Predict on the new data
    y_pred_new = final_model.predict([X1_unselected_selected, X2_unselected_selected, X3_unselected_selected])
    
    # Compute Spearman correlation
    spearman_corr_new, _ = spearmanr(y_unselected_selected, y_pred_new)
    spearman_array.append(spearman_corr_new)
    print(f"Spearman Correlation (new data, trial {i+1}): {spearman_corr_new}")

for spearman_corr in spearman_array:
    print(spearman_corr)

In [ ]:
# CNN1+CNN2+MLP
from tensorflow import keras

import tensorflow as tf
from keras import layers, models
import keras_tuner as kt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr
from keras.callbacks import EarlyStopping

# Custom callback to save overall best model across all trials
class OverallBestModelCheckpoint(tf.keras.callbacks.Callback):
    overall_best_val_loss = np.Inf  # Class-level variable to track global best

    def __init__(self, filepath, monitor='val_loss', mode='min', verbose=0):
        super(OverallBestModelCheckpoint, self).__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.verbose = verbose
        self.best_val_loss_in_trial = np.Inf  # Best in current trial

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_val_loss = logs.get(self.monitor)
        if current_val_loss is None:
            return
        
        # Update the best value for the current trial if improved
        if current_val_loss < self.best_val_loss_in_trial:
            self.best_val_loss_in_trial = current_val_loss
        
        # Compare with overall best across trials
        if self.best_val_loss_in_trial < OverallBestModelCheckpoint.overall_best_val_loss:
            OverallBestModelCheckpoint.overall_best_val_loss = self.best_val_loss_in_trial
            self.model.save(self.filepath)
            if self.verbose > 0:
                print(f'\nEpoch {epoch+1}: {self.monitor} improved to {current_val_loss:.5f} (overall), model saved.')

def build_fusion_model(hp):
    """
    Builds a three-branch CNN+MLP model using the Keras Functional API.
    Each branch processes one input, and then the outputs are concatenated
    for a final prediction (regression).
    """

    # Branch 1 (CNN) for input of shape (112, 4)
    input1 = keras.Input(shape=(112, 4), name='branch1_input')

    x1 = layers.Conv1D(
        filters=hp.Int('filters1', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size1', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input1)
    x1 = layers.MaxPooling1D(pool_size=2)(x1)
    x1 = layers.Flatten()(x1)
    x1 = layers.Dense(
        hp.Int('dense_units1', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x1)

    # Branch 2 (CNN) for input of shape (184, 3)
    input2 = keras.Input(shape=(184, 3), name='branch2_input')

    x2 = layers.Conv1D(
        filters=hp.Int('filters2', min_value=16, max_value=128, step=16),
        kernel_size=hp.Int('kernel_size2', min_value=2, max_value=7, step=1),
        activation='relu',
        padding='same'
    )(input2)
    x2 = layers.MaxPooling1D(pool_size=2)(x2)
    x2 = layers.Flatten()(x2)
    x2 = layers.Dense(
        hp.Int('dense_units2', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x2)

    # Branch 3 (MLP) for input of shape (24,)
    input3 = keras.Input(shape=(24,), name='branch3_input')

    x3 = layers.Dense(
        hp.Int('dense_units3', min_value=16, max_value=256, step=16),
        activation='relu'
    )(input3)
    
    # Branch 4 for input of shape (1,)
    input4 = keras.Input(shape=(1,), name='branch4_input')

    x4 = layers.Dense(
        hp.Int('mlp_units', min_value=8, max_value=64, step=8),
        activation='relu'
    )(input4)

    # Fuse (concatenate) the outputs of all three branches
    merged = layers.Concatenate()([x1, x2, x3, x4])

    # Add dropout layer with hyperparameter dropout_rate ranging from 0 to 0.5
    dropout_rate = hp.Float('dropout_rate', min_value=0.0, max_value=0.5)
    x = layers.Dropout(rate=dropout_rate)(merged)

    x = layers.Dense(
        hp.Int('dense_units_merged', min_value=16, max_value=256, step=16),
        activation='relu'
    )(x)

    # Final output layer for regression (1 unit)
    output = layers.Dense(1, name='output')(x)

    # Build and compile the final model
    model = keras.Model(inputs=[input1, input2, input3, input4], outputs=output)

    # Compile the model with AdamW optimizer (L2 regularization via weight decay = 1e-4) and mean squared error loss
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG'),
        weight_decay=1e-4
    )
    model.compile(
        optimizer=optimizer,
        loss='mean_squared_error',
        metrics=['mae']
    )

    return model

# Hyperparameter Tuner Setup
tuner_fusion = kt.Hyperband(
    build_fusion_model,
    objective='val_loss',  # Minimize validation loss for regression
    max_epochs=100,
    factor=3,  # Hyperband factor
    directory='cnn1_cnn2_mlp_tuning',
    project_name='cnn1_cnn2_mlp_hyperparameter_search_regression'
)

# Data Loading and Preprocessing
# 1) Load Branch 1 data: shape (n_samples, 112, 4)
with open("Feature_CNN1.txt", 'r') as file:
    data_branch1 = []
    current_array = []
    for line in file:
        line = line.strip()
        if line == "":
            if current_array:
                data_branch1.append(current_array)
                current_array = []
        else:
            line = line.replace('[', '').replace(']', '')
            current_array.append([float(x.strip().replace(',', '')) for x in line.split()])
    if current_array:
        data_branch1.append(current_array)

X_branch1 = np.array(data_branch1)  # shape: (n_samples, 112, 4)
X_branch1 = X_branch1.reshape(len(X_branch1), 112, 4)  

# 2) Load Branch 2 data: shape (n_samples, 184, 3)
with open("Feature_CNN2.txt", 'r') as file:
    data_branch2 = []
    current_array2 = []
    for line in file:
        line = line.strip()
        if line == "":
            if current_array2:
                data_branch2.append(current_array2)

                current_array2 = []
        else:
            line = line.replace('[', '').replace(']', '')
            current_array2.append([float(x.strip().replace(',', '')) for x in line.split()])
    if current_array2:
        data_branch2.append(current_array2)

X_branch2 = np.array(data_branch2)  # shape: (n_samples, 184, 3)
X_branch2 = X_branch2.reshape(len(X_branch2), 184, 3)  

# 3) Load Branch 3 data (MLP) => shape (n_samples, 24)
X_branch3 = []
with open("Feature_MLP.txt", 'r') as file:
    for line in file:
        line = line.strip()
        values = line.replace('[', '').replace(']', '').split()
        float_vals = [float(v.strip().replace(',', '')) for v in values]
        X_branch3.append(float_vals)

X_branch3 = np.array(X_branch3)  # shape: (n_samples, 24)
X_branch3 = X_branch3.reshape(len(X_branch3), 24)  

# 4) Read MLP input from merged_time.txt (each sample has one value)
data_branch4 = []
with open('merged_Cas13a_1e9_time_before_19min.txt', 'r') as file:
    data_branch4 = [float(line.strip()) for line in file.readlines()]
X_branch4 = np.array(data_branch4)
X_branch4 = X_branch4.reshape(len(X_branch4), 1)  

# 5) Load regression targets (y)
rate = []
with open('merged_Cas13a_1e9_values_unique_before_19min.txt', 'r') as file:
    rate = [float(line.strip()) for line in file.readlines()]
y = np.array(rate)
    
# Check shape consistency
assert len(X_branch1) == len(X_branch2) == len(X_branch3) == len(X_branch4) == len(y), \
    "All inputs must have the same number of samples."

np.random.seed(42)
indices_to_select = np.random.choice(len(X_branch1), size=800, replace=False)
all_indices = np.arange(len(X_branch1))
indices_unselected = np.setdiff1d(all_indices, indices_to_select)

X1_selected = X_branch1[indices_to_select]
X2_selected = X_branch2[indices_to_select]
X3_selected = X_branch3[indices_to_select]
X4_selected = X_branch4[indices_to_select]
y_selected  = y[indices_to_select]

X1_unselected = X_branch1[indices_unselected]
X2_unselected = X_branch2[indices_unselected]
X3_unselected = X_branch3[indices_unselected]
X4_unselected = X_branch4[indices_unselected]
y_unselected  = y[indices_unselected]

# Split the selected data into training and testing sets
X1_train, X1_test, X2_train, X2_test, X3_train, X3_test, X4_train, X4_test, y_train, y_test = train_test_split(
    X1_selected, 
    X2_selected,
    X3_selected,
    X4_selected,
    y_selected,
    test_size=0.2,
    random_state=42
)

# Define callbacks: EarlyStopping, TensorBoard, etc.
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Instantiate the custom overall best model checkpoint callback
overall_best_checkpoint = OverallBestModelCheckpoint(
    filepath='best_CNN1_CNN2_MLP.h5',
    monitor='val_loss',
    mode='min',
    verbose=1
)

log_dir = "logs_best_CNN1_CNN2_MLP"
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# Run Hyperparameter Search
tuner_fusion.search(
    [X1_train, X2_train, X3_train, X4_train],  
    y_train,
    validation_data=([X1_test, X2_test, X3_test, X4_test], y_test),
    callbacks=[early_stopping, tensorboard_callback, overall_best_checkpoint]
)

# Load the saved overall best model for final evaluation.
final_model = tf.keras.models.load_model('best_CNN1_CNN2_MLP.h5')

# Retrieve the Best Model and Evaluate
val_loss, val_mae = final_model.evaluate([X1_test, X2_test, X3_test, X4_test], y_test)
print(f"\nBest Fusion CNN+MLP model - Test Loss: {val_loss}, Test MAE: {val_mae}")

# Predict and compute Spearman correlation on the test set
y_pred_test = final_model.predict([X1_test, X2_test, X3_test, X4_test])
spearman_corr_test, _ = spearmanr(y_test, y_pred_test)
print(f"Spearman Correlation (test set): {spearman_corr_test}\n")

# Evaluate on the unselected data multiple times 
spearman_array = []
for i in range(100):  
    # Randomly select e.g. 1000 samples from the unselected set
    if len(X1_unselected) < 100:
        # If unselected < 100, either skip or just take them all
        size_to_take = len(X1_unselected)
    else:
        size_to_take = 100

    indices_to_select_new = np.random.choice(len(X1_unselected), size=size_to_take, replace=False)

    X1_unselected_selected = X1_unselected[indices_to_select_new]
    X2_unselected_selected = X2_unselected[indices_to_select_new]
    X3_unselected_selected = X3_unselected[indices_to_select_new]
    X4_unselected_selected = X4_unselected[indices_to_select_new]
    y_unselected_selected  = y_unselected[indices_to_select_new]

    # Predict on the new data
    y_pred_new = final_model.predict([X1_unselected_selected, 
                                                   X2_unselected_selected,
                                                   X3_unselected_selected,
                                                   X4_unselected_selected])
    
    # Compute Spearman correlation
    spearman_corr_new, _ = spearmanr(y_unselected_selected, y_pred_new)
    spearman_array.append(spearman_corr_new)
    print(f"Spearman Correlation (new data, trial {i+1}): {spearman_corr_new}")

for spearman_corr in spearman_array:
    print(spearman_corr)